# Guide complet du Module tsforecast.utils

Ce notebook illustre les fonctionnalités du module `tsforecast.utils`, qui fournit des outils pour la manipulation, la validation et la transformation de données temporelles et de panel.

## Table des Matières

1. **Introduction au Module Utils**
   - Architecture générale
   - Normalizers, Converters et Validators
   
2. **Manipulation de Temps** (`time.py`)
   - Identification des limites de périodes
   - Conversion string ↔ datetime
   
3. **Validation de Données** (`validation.py`)
   - Validation de séries temporelles et panel
   - Restauration de structure
   - Vérifications groupées
   
4. **Gestion des Fréquences** (`frequency/`)
   - Normalisation de fréquences
   - Conversion et agrégation
   - Interpolation et alignement
   
5. **Gestion des Durées** (`duration/`)
   - Normalisation de durées
   - Conversion entre unités
   
6. **Positions de Périodes** (`position/`)
   - Positions start/end
   - Manipulation d'offsets pandas
   
7. **Transformers de Base** (aperçu)
   - Mixins disponibles
   - Intégration sklearn
   
8. **Exemple Complet End-to-End**
   - Workflow réaliste multi-composants

## 1. Introduction au Module Utils

Le module `tsforecast.utils` fournit une suite d'outils pour travailler avec des données temporelles et de panel. Il s'organise autour de trois concepts principaux:

### Architecture

- **Normalizers** (`TemporalNormalizer`): Standardisent les représentations (fréquences, durées, positions)
- **Converters** (`TemporalConverter`): Transforment les valeurs entre différentes unités
- **Validators**: Vérifient et préparent les données temporelles

### Sous-modules principaux

- `time`: Manipulation de dates et périodes
- `validation`: Validation de structure des données
- `frequency/`: Gestion des fréquences temporelles
- `duration/`: Gestion des durées
- `position/`: Gestion des positions dans les périodes

### Importation des Modules

In [1]:
# Modules de base
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Utilitaires - time
from tsforecast.utils.time import (
    get_period_start, 
    get_period_end, 
    get_period_boundaries,
    timeseries_to_string, 
    string_to_timeseries
)

# Utilitaires - validation
from tsforecast.utils.validation import (
    validate_temporal_data,
    restore_original_structure,
    validate_entities_grouped,
    validate_sorted_within_groups
)

# Utilitaires - frequency
from tsforecast.utils.frequency import (
    FrequencyNormalizer,
    FrequencyConverter,
    normalize_frequency,
    convert_frequency,
    to_literal as freq_to_literal,
    is_higher_frequency
)

# Utilitaires - duration
from tsforecast.utils.duration import (
    DurationNormalizer,
    DurationConverter,
    normalize_duration,
    convert_duration,
    to_literal as duration_to_literal
)

# Utilitaires - position
from tsforecast.utils.position import (
    PeriodPositionNormalizer,
    PeriodPositionConverter
)

print("✓ Tous les modules importés avec succès")

✓ Tous les modules importés avec succès


## 2. Manipulation de Temps

Le module `time.py` fournit des fonctions pour identifier les limites de périodes et les convertir entre formats datetime et string.

### 2.1 Identification des Limites de Périodes

Les fonctions `get_period_start()`, `get_period_end()` et `get_period_boundaries()` permettent d'identifier les dates de début et de fin d'une période selon différentes fréquences.

In [2]:
# Date de référence (un mercredi au milieu du mois)
date = pd.Timestamp('2024-03-15 14:30:00')
print(f"Date de référence: {date} ({date.day_name()})\n")
print("="*60)

# Fréquences standards
print("\n📅 FRÉQUENCES STANDARDS")
print("-"*60)
for frequency in ['daily', 'weekly', 'monthly', 'quarterly', 'annual']:
    start = get_period_start(date=date, frequency=frequency)
    end = get_period_end(date=date, frequency=frequency)
    print(f"{frequency:12s} | Début: {start} | Fin: {end}")

# Fréquences atypiques (impact significatif)
print("\n⚙️ FRÉQUENCES ATYPIQUES (comportement spécifique)")
print("-"*60)

# Business day: exclut week-ends
start_bd = get_period_start(date=date, frequency='business_daily')
end_bd = get_period_end(date=date, frequency='business_daily')
print(f"{'business_day':12s} | Début: {start_bd} | Fin: {end_bd}")
print(f"  → Même jour si jour ouvré (ignore le week-end)")

# Semi-monthly: deux périodes par mois (1-15, 16-fin)
date_early = pd.Timestamp('2024-03-05')
date_late = pd.Timestamp('2024-03-20')
start_sm1 = get_period_start(date=date_early, frequency='semi_monthly')
end_sm1 = get_period_end(date=date_early, frequency='semi_monthly')
start_sm2 = get_period_start(date=date_late, frequency='semi_monthly')
end_sm2 = get_period_end(date=date_late, frequency='semi_monthly')
print(f"{'semi_monthly':12s} | Date: {date_early.date()} → [{start_sm1.date()}, {end_sm1.date()}]")
print(f"{'':12s} | Date: {date_late.date()} → [{start_sm2.date()}, {end_sm2.date()}]")

# Fréquences hautes résolutions (mentionnées)
print("\n🕐 Fréquences haute résolution disponibles: hourly, minute, second, millisecond, etc.")

# Utilisation de get_period_boundaries()
print("\n🎯 UTILISATION DE get_period_boundaries()")
print("-"*60)
start, end = get_period_boundaries(date=date, frequency='monthly')
print(f"Période mensuelle pour {date.date()}:")
print(f"  Boundaries: ({start.date()}, {end.date()})")
print(f"  Durée: {(end - start).days + 1} jours")

Date de référence: 2024-03-15 14:30:00 (Friday)


📅 FRÉQUENCES STANDARDS
------------------------------------------------------------
daily        | Début: 2024-03-15 00:00:00 | Fin: 2024-03-16 00:00:00
weekly       | Début: 2024-03-11 00:00:00 | Fin: 2024-03-18 00:00:00
monthly      | Début: 2024-03-01 00:00:00 | Fin: 2024-04-01 00:00:00
quarterly    | Début: 2024-01-01 00:00:00 | Fin: 2024-04-01 00:00:00
annual       | Début: 2024-01-01 00:00:00 | Fin: 2025-01-01 00:00:00

⚙️ FRÉQUENCES ATYPIQUES (comportement spécifique)
------------------------------------------------------------
business_day | Début: 2024-03-15 00:00:00 | Fin: 2024-03-16 00:00:00
  → Même jour si jour ouvré (ignore le week-end)
semi_monthly | Date: 2024-03-05 → [2024-03-01, 2024-03-16]
             | Date: 2024-03-20 → [2024-03-16, 2024-04-01]

🕐 Fréquences haute résolution disponibles: hourly, minute, second, millisecond, etc.

🎯 UTILISATION DE get_period_boundaries()
---------------------------------------------

### 2.2 Conversion entre Datetime et String

Les fonctions `timeseries_to_string()` et `string_to_timeseries()` permettent de convertir l'index d'une série entre les formats datetime et string, utiles pour l'export/import et le stockage.

### Conversion d'une série temporelle en chaînes de caractères

In [3]:
# Création d'une série temporelle de démonstration
dates = pd.date_range('2023-01-01', periods=8, freq='MS')  # Mensuel début de mois
values = np.random.randn(8).cumsum() + 100
ts_demo = pd.Series(values, index=dates, name='economic_indicator')

print("📊 Série temporelle originale:")
print(ts_demo)
print(f"\nType de l'index: {type(ts_demo.index[0])}")

print("\n" + "="*50)

# Conversion avec format par défaut (année-mois-jour)
ts_string_default = timeseries_to_string(ts_demo)
print("🔤 Conversion avec format par défaut (%Y-%m-%d):")
print(ts_string_default)
print(f"Type de l'index: {type(ts_string_default.index[0])}")

print("\n" + "="*30)

# Conversion avec format personnalisé
ts_string_custom = timeseries_to_string(ts_demo, format="%B %Y")  # Mois complet Année
print("🔤 Conversion avec format personnalisé (%B %Y):")
print(ts_string_custom)

print("\n" + "="*30)

# Conversion avec format français
ts_string_fr = timeseries_to_string(ts_demo, format="%d/%m/%Y")  # Format français
print("🔤 Conversion au format français (%d/%m/%Y):")
print(ts_string_fr)

print("\n" + "="*30)

# Conversion avec format ISO complet
ts_string_iso = timeseries_to_string(ts_demo, format="%Y-%m-%dT%H:%M:%S")
print("🔤 Conversion au format ISO complet:")
print(ts_string_iso.head(3))  # Affichage partiel pour la lisibilité

📊 Série temporelle originale:
2023-01-01     99.502883
2023-02-01     98.772441
2023-03-01    100.532038
2023-04-01     99.519625
2023-05-01     97.086834
2023-06-01     98.331747
2023-07-01     98.509859
2023-08-01     96.402554
Freq: MS, Name: economic_indicator, dtype: float64

Type de l'index: <class 'pandas._libs.tslibs.timestamps.Timestamp'>

🔤 Conversion avec format par défaut (%Y-%m-%d):
2023-01-01     99.502883
2023-02-01     98.772441
2023-03-01    100.532038
2023-04-01     99.519625
2023-05-01     97.086834
2023-06-01     98.331747
2023-07-01     98.509859
2023-08-01     96.402554
Name: economic_indicator, dtype: float64
Type de l'index: <class 'str'>

🔤 Conversion avec format personnalisé (%B %Y):
January 2023      99.502883
February 2023     98.772441
March 2023       100.532038
April 2023        99.519625
May 2023          97.086834
June 2023         98.331747
July 2023         98.509859
August 2023       96.402554
Name: economic_indicator, dtype: float64

🔤 Conversion au

### Conversion inverse : chaînes de caractères vers série temporelle

In [4]:
# Conversion inverse sans spécification de format (inférence automatique)
print("🔄 Conversion inverse avec inférence automatique:")
ts_converted_auto = string_to_timeseries(ts_string_default)
print("Série avec format par défaut reconvertie:")
print(ts_converted_auto)
print(f"Type de l'index: {type(ts_converted_auto.index[0])}")

# Vérification que nous retrouvons les mêmes données
print(f"\n✅ Données identiques après conversion: {ts_demo.equals(ts_converted_auto)}")

print("\n" + "="*50)

# Conversion inverse avec format spécifique
print("🔄 Conversion inverse avec format spécifique:")
ts_converted_specific = string_to_timeseries(ts_string_fr, format="%d/%m/%Y")
print("Série avec format français reconvertie:")
print(ts_converted_specific)

# Vérification de l'équivalence (en ignorant les heures/minutes)
dates_equal = all(ts_demo.index.date == ts_converted_specific.index.date)
values_equal = np.allclose(ts_demo.values, ts_converted_specific.values)
print(f"\n✅ Dates équivalentes: {dates_equal}")
print(f"✅ Valeurs équivalentes: {values_equal}")

print("\n" + "="*50)

# Démonstration avec des formats complexes
print("🎯 Démonstration avec des formats plus complexes:")

# Création d'une série avec dates irrégulières
irregular_dates = ['2023-01-15', '2023-02-28', '2023-04-10', '2023-07-04', '2023-12-25']
irregular_values = [150.2, 148.7, 152.1, 149.5, 155.8]
ts_irregular = pd.Series(irregular_values, index=irregular_dates, name='irregular_series')

print("Série avec index string (irrégulier):")
print(ts_irregular)
print(f"Type de l'index: {type(ts_irregular.index[0])}")

# Conversion en datetime
ts_datetime = string_to_timeseries(ts_irregular)
print("\nAprès conversion en datetime:")
print(ts_datetime)
print(f"Type de l'index: {type(ts_datetime.index[0])}")

print("\n" + "="*30)

# Test avec format américain
american_dates = ['01/15/2023', '02/28/2023', '04/10/2023']
american_values = [100, 105, 98]
ts_american = pd.Series(american_values, index=american_dates)

print("Série avec format américain (MM/dd/yyyy):")
print(ts_american)

# Conversion avec format spécifique
ts_american_converted = string_to_timeseries(ts_american, format="%m/%d/%Y")
print("Après conversion avec format spécifique:")
print(ts_american_converted)

# Comparaison avec inférence automatique
ts_american_inferred = string_to_timeseries(ts_american)
print("Avec inférence automatique:")
print(ts_american_inferred)

# Vérification des différences potentielles
dates_same = ts_american_converted.index.equals(ts_american_inferred.index)
print(f"\n⚠️  Résultats identiques: {dates_same}")
if not dates_same:
    print("Attention: L'inférence automatique peut parfois interpréter différemment les dates ambigües!")

🔄 Conversion inverse avec inférence automatique:
Série avec format par défaut reconvertie:
2023-01-01     99.502883
2023-02-01     98.772441
2023-03-01    100.532038
2023-04-01     99.519625
2023-05-01     97.086834
2023-06-01     98.331747
2023-07-01     98.509859
2023-08-01     96.402554
Name: economic_indicator, dtype: float64
Type de l'index: <class 'pandas._libs.tslibs.timestamps.Timestamp'>

✅ Données identiques après conversion: True

🔄 Conversion inverse avec format spécifique:
Série avec format français reconvertie:
2023-01-01     99.502883
2023-02-01     98.772441
2023-03-01    100.532038
2023-04-01     99.519625
2023-05-01     97.086834
2023-06-01     98.331747
2023-07-01     98.509859
2023-08-01     96.402554
Name: economic_indicator, dtype: float64

✅ Dates équivalentes: True
✅ Valeurs équivalentes: True

🎯 Démonstration avec des formats plus complexes:
Série avec index string (irrégulier):
2023-01-15    150.2
2023-02-28    148.7
2023-04-10    152.1
2023-07-04    149.5
202

### Cas d'usage pratiques et exemples avancés

In [5]:
# Cas d'usage 1: Sauvegarde et chargement de données
print("💾 Cas d'usage 1: Sauvegarde et chargement de données")
print("="*55)

# Simulation d'une série de données économiques
dates_econ = pd.date_range('2020-01-01', '2024-12-01', freq='QS')  # Trimestriel
pib_growth = np.random.normal(0.5, 2.0, len(dates_econ)).cumsum()
ts_pib = pd.Series(pib_growth, index=dates_econ, name='pib_growth_rate')

print("Série PIB originale (5 premières valeurs):")
print(ts_pib.head())

# Conversion pour sauvegarde dans un format lisible
ts_pib_string = timeseries_to_string(ts_pib, format="%Y-Q%q")  # Format trimestriel
print("\nFormat string pour sauvegarde:")
print(ts_pib_string.head())

# Simulation: sauvegarde et rechargement (DataFrame pour démonstration)
df_save = pd.DataFrame({
    'date_str': ts_pib_string.index,
    'pib_growth': ts_pib_string.values
})
print("\nDataFrame pour sauvegarde CSV:")
print(df_save.head())

# Rechargement et reconversion
ts_loaded = pd.Series(df_save['pib_growth'].values, index=df_save['date_str'])
ts_reloaded = string_to_timeseries(ts_loaded)
print("\nSérie rechargée:")
print(ts_reloaded.head())

print("\n" + "="*40)

# Cas d'usage 2: Comparaison de différents formats de dates
print("📅 Cas d'usage 2: Harmonisation de formats de dates multiples")
print("="*65)

# Simulation de données provenant de sources différentes
# Source 1: Format ISO
dates_iso = ['2023-01-01', '2023-02-01', '2023-03-01']
values_iso = [100, 102, 104]
ts_iso = pd.Series(values_iso, index=dates_iso, name='source_iso')

# Source 2: Format français
dates_fr = ['01/04/2023', '01/05/2023', '01/06/2023']
values_fr = [106, 108, 110]
ts_fr_source = pd.Series(values_fr, index=dates_fr, name='source_fr')

# Source 3: Format américain
dates_us = ['07/01/2023', '08/01/2023', '09/01/2023']
values_us = [112, 114, 116]
ts_us_source = pd.Series(values_us, index=dates_us, name='source_us')

print("Données provenant de 3 sources différentes:")
print("Source ISO:", ts_iso.index.tolist())
print("Source FR:", ts_fr_source.index.tolist())
print("Source US:", ts_us_source.index.tolist())

# Harmonisation vers datetime
ts_iso_dt = string_to_timeseries(ts_iso)
ts_fr_dt = string_to_timeseries(ts_fr_source, format="%d/%m/%Y")
ts_us_dt = string_to_timeseries(ts_us_source, format="%m/%d/%Y")

print("\nAprès harmonisation:")
print("Source ISO:", ts_iso_dt.index.tolist()[:2])
print("Source FR:", ts_fr_dt.index.tolist()[:2])
print("Source US:", ts_us_dt.index.tolist()[:2])

# Concaténation des séries harmonisées
ts_combined = pd.concat([ts_iso_dt, ts_fr_dt, ts_us_dt]).sort_index()
print("\nSérie combinée et triée:")
print(ts_combined)

print("\n" + "="*40)

# Cas d'usage 3: Analyse de périodicité et conversion de fréquence
print("🔄 Cas d'usage 3: Impact des formats sur l'analyse temporelle")
print("="*60)

# Création d'une série avec différentes résolutions temporelles
hourly_dates = pd.date_range('2023-01-01 00:00', '2023-01-01 23:00', freq='H')
hourly_data = np.sin(np.arange(len(hourly_dates)) * 2 * np.pi / 24) + np.random.normal(0, 0.1, len(hourly_dates))
ts_hourly = pd.Series(hourly_data, index=hourly_dates)

print(f"Série horaire (24 observations): {len(ts_hourly)} points")

# Conversion vers différents formats de granularité
formats_and_names = [
    ("%Y-%m-%d %H:00", "Format heure"),
    ("%Y-%m-%d", "Format jour"),
    ("%Y-%m", "Format mois"),
    ("%Y", "Format année")
]

for format_str, name in formats_and_names:
    ts_string_format = timeseries_to_string(ts_hourly, format=format_str)
    unique_dates = len(ts_string_format.index.unique())
    print(f"{name:12}: {unique_dates:2d} périodes uniques, exemple: '{ts_string_format.index[0]}'")

print("\n" + "="*40)

# Cas d'usage 4: Validation et gestion d'erreurs
print("⚠️  Cas d'usage 4: Gestion des cas limites")
print("="*45)

# Test avec des dates ambiguës
ambiguous_dates = ['01/02/2023', '03/04/2023', '05/06/2023']
ambiguous_values = [1, 2, 3]
ts_ambiguous = pd.Series(ambiguous_values, index=ambiguous_dates)

print("Dates ambiguës (jour/mois ou mois/jour?):")
print(ts_ambiguous)

# Conversion avec formats différents
try:
    ts_dm = string_to_timeseries(ts_ambiguous, format="%d/%m/%Y")
    print("\nInterprétation jour/mois:")
    print(ts_dm)
except Exception as e:
    print(f"Erreur format jour/mois: {e}")

try:
    ts_md = string_to_timeseries(ts_ambiguous, format="%m/%d/%Y")
    print("\nInterprétation mois/jour:")
    print(ts_md)
except Exception as e:
    print(f"Erreur format mois/jour: {e}")

print("\n📋 Résumé des bonnes pratiques:")
print("1. Toujours spécifier le format lors de la conversion inverse si ambiguïté")
print("2. Vérifier la cohérence des dates après conversion")
print("3. Utiliser des formats standards (ISO 8601) si possible")
print("4. Tester avec des échantillons de données avant le traitement en masse")

💾 Cas d'usage 1: Sauvegarde et chargement de données
Série PIB originale (5 premières valeurs):
2020-01-01   -1.586094
2020-04-01   -4.151503
2020-07-01   -0.516001
2020-10-01   -1.862984
2021-01-01   -0.607213
Freq: QS-JAN, Name: pib_growth_rate, dtype: float64

Format string pour sauvegarde:
2020-01-01 00:00:00   -1.586094
2020-04-01 00:00:00   -4.151503
2020-07-01 00:00:00   -0.516001
2020-10-01 00:00:00   -1.862984
2021-01-01 00:00:00   -0.607213
Name: pib_growth_rate, dtype: float64

DataFrame pour sauvegarde CSV:
              date_str  pib_growth
0  2020-01-01 00:00:00   -1.586094
1  2020-04-01 00:00:00   -4.151503
2  2020-07-01 00:00:00   -0.516001
3  2020-10-01 00:00:00   -1.862984
4  2021-01-01 00:00:00   -0.607213

Série rechargée:
date_str
2020-01-01   -1.586094
2020-04-01   -4.151503
2020-07-01   -0.516001
2020-10-01   -1.862984
2021-01-01   -0.607213
dtype: float64

📅 Cas d'usage 2: Harmonisation de formats de dates multiples
Données provenant de 3 sources différentes:
So

## 3. Validation de Données

Le module `validation.py` fournit des outils pour valider et préparer les données temporelles et de panel, en s'assurant qu'elles respectent la structure attendue par les autres modules du package.

### 3.1 Validation de Séries Temporelles avec `validate_temporal_data()`

Cette fonction valide et prépare les données en convertissant les colonnes temporelles en index, en vérifiant différentes contraintes.

In [6]:
# Exemple 1: Validation d'une Series avec index datetime (déjà valide)
print("📊 Exemple 1: Series avec index datetime")
print("="*60)
dates = pd.date_range('2023-01-01', periods=5, freq='M')
values = [100, 105, 103, 108, 112]
ts_valid = pd.Series(values, index=dates, name='sales')
print("Données originales:")
print(ts_valid)

validated = validate_temporal_data(ts_valid)
print("\nAprès validation (identique):")
print(validated)
print(f"✓ Type d'index: {type(validated.index)}")

print("\n" + "="*60)

# Exemple 2: DataFrame avec colonne temporelle → conversion en index
print("\n📊 Exemple 2: DataFrame avec colonne 'date' → conversion en index")
print("="*60)
df = pd.DataFrame({
    'date': ['2023-01-01', '2023-02-01', '2023-03-01', '2023-04-01'],
    'sales': [100, 105, 103, 108],
    'costs': [80, 82, 79, 84]
})
print("Données originales:")
print(df)
print(f"Index original: {df.index.tolist()}")

# Validation avec conversion de la colonne 'date' en index
validated_df = validate_temporal_data(df, time_col='date')
print("\nAprès validation:")
print(validated_df)
print(f"✓ Colonne 'date' convertie en index datetime")
print(f"✓ Type d'index: {type(validated_df.index)}")

print("\n" + "="*60)

# Exemple 3: Impact du paramètre strict (True vs False)
print("\n⚙️ Exemple 3: Paramètre strict - impact sur les erreurs")
print("="*60)

# Données non triées (problème potentiel)
df_unsorted = pd.DataFrame({
    'date': ['2023-03-01', '2023-01-01', '2023-02-01'],  # Non triées!
    'value': [30, 10, 20]
})
print("Données non triées par date:")
print(df_unsorted)

# Mode strict=False (warning seulement)
print("\n🟡 Avec strict=False (warning):")
validated_non_strict = validate_temporal_data(df_unsorted, time_col='date', strict=False, sort_data=False)
print(validated_non_strict)
print("→ Données acceptées avec avertissement")

# Mode strict=False avec sort_data=True (corrige automatiquement)
print("\n🟢 Avec strict=False et sort_data=True:")
validated_sorted = validate_temporal_data(df_unsorted, time_col='date', strict=False, sort_data=True)
print(validated_sorted)
print("✓ Données automatiquement triées")

print("\nℹ️ Note: strict=True génèrerait une erreur pour des données non triées")

📊 Exemple 1: Series avec index datetime
Données originales:
2023-01-31    100
2023-02-28    105
2023-03-31    103
2023-04-30    108
2023-05-31    112
Freq: ME, Name: sales, dtype: int64

Après validation (identique):
2023-01-31    100
2023-02-28    105
2023-03-31    103
2023-04-30    108
2023-05-31    112
Freq: ME, Name: sales, dtype: int64
✓ Type d'index: <class 'pandas.core.indexes.datetimes.DatetimeIndex'>


📊 Exemple 2: DataFrame avec colonne 'date' → conversion en index
Données originales:
         date  sales  costs
0  2023-01-01    100     80
1  2023-02-01    105     82
2  2023-03-01    103     79
3  2023-04-01    108     84
Index original: [0, 1, 2, 3]

Après validation:
            sales  costs
date                    
2023-01-01    100     80
2023-02-01    105     82
2023-03-01    103     79
2023-04-01    108     84
✓ Colonne 'date' convertie en index datetime
✓ Type d'index: <class 'pandas.core.indexes.datetimes.DatetimeIndex'>


⚙️ Exemple 3: Paramètre strict - impact sur l

### 3.2 Validation de Données Panel (Multi-entités)

Pour les données panel, `validate_temporal_data()` peut gérer plusieurs entités avec le paramètre `panel_cols`.

In [7]:
# Données panel: plusieurs entités avec séries temporelles
print("📊 Données Panel: Validation multi-entités")
print("="*60)

# Création de données panel
df_panel = pd.DataFrame({
    'country': ['FR', 'FR', 'FR', 'DE', 'DE', 'DE', 'IT', 'IT', 'IT'],
    'date': ['2023-01', '2023-02', '2023-03'] * 3,
    'gdp': [100, 102, 104, 200, 203, 206, 150, 151, 153],
    'inflation': [2.1, 2.3, 2.2, 1.8, 1.9, 2.0, 3.2, 3.1, 3.0]
})
print("Données panel originales:")
print(df_panel)
print(f"\nStructure: {len(df_panel)} observations, {df_panel['country'].nunique()} pays")

# Validation avec panel_cols
print("\n🔄 Validation avec panel_cols=['country']:")
validated_panel = validate_temporal_data(
    df_panel, 
    time_col='date', 
    panel_cols=['country'],
    sort_data=True
)
print(validated_panel)
print(f"\n✓ Index multi-niveau (country, date): {type(validated_panel.index)}")
print(f"✓ Niveaux de l'index: {validated_panel.index.names}")

# Accès aux données d'un pays spécifique
print("\n📍 Accès aux données de la France:")
print(validated_panel.loc['FR'])

📊 Données Panel: Validation multi-entités
Données panel originales:
  country     date  gdp  inflation
0      FR  2023-01  100        2.1
1      FR  2023-02  102        2.3
2      FR  2023-03  104        2.2
3      DE  2023-01  200        1.8
4      DE  2023-02  203        1.9
5      DE  2023-03  206        2.0
6      IT  2023-01  150        3.2
7      IT  2023-02  151        3.1
8      IT  2023-03  153        3.0

Structure: 9 observations, 3 pays

🔄 Validation avec panel_cols=['country']:
                    gdp  inflation
country date                      
DE      2023-01-01  200        1.8
        2023-02-01  203        1.9
        2023-03-01  206        2.0
FR      2023-01-01  100        2.1
        2023-02-01  102        2.3
        2023-03-01  104        2.2
IT      2023-01-01  150        3.2
        2023-02-01  151        3.1
        2023-03-01  153        3.0

✓ Index multi-niveau (country, date): <class 'pandas.core.indexes.multi.MultiIndex'>
✓ Niveaux de l'index: ['country',

### 3.3 Restauration de Structure avec `restore_original_structure()`

Après traitement, on peut restaurer la structure originale des données en utilisant les métadonnées sauvegardées.

In [8]:
# Workflow complet: validation → traitement → restauration
print("🔄 Workflow: Validation → Traitement → Restauration")
print("="*60)

# Données originales avec structure spécifique
df_original = pd.DataFrame({
    'timestamp': ['2023-01-01', '2023-02-01', '2023-03-01', '2023-04-01'],
    'revenue': [1000, 1100, 1050, 1200],
    'users': [50, 55, 52, 60]
})
print("1️⃣ Données originales:")
print(df_original)
print(f"   Index: {df_original.index.tolist()}")
print(f"   Colonnes: {df_original.columns.tolist()}")

# Validation avec métadonnées
print("\n2️⃣ Validation avec return_metadata=True:")
validated, metadata = validate_temporal_data(
    df_original, 
    time_col='timestamp',
    return_metadata=True
)
print(validated)
print(f"   ✓ 'timestamp' est maintenant l'index")
print(f"   Métadonnées sauvegardées: {list(metadata.keys())}")

# Simulation de traitement (ex: calcul de variations)
print("\n3️⃣ Traitement (calcul de variations):")
processed = validated.copy()
processed['revenue_change'] = processed['revenue'].pct_change() * 100
processed['users_change'] = processed['users'].pct_change() * 100
print(processed)

# Restauration de la structure originale
print("\n4️⃣ Restauration de la structure originale:")
restored = restore_original_structure(processed, metadata)
print(restored)
print(f"   ✓ Colonne 'timestamp' restaurée")
print(f"   ✓ Index original restauré: {restored.index.tolist()}")
print(f"   ✓ Ordre des colonnes préservé (+ nouvelles colonnes ajoutées)")

print("\n✅ Avantage: Permet de travailler avec un index temporel pendant le traitement,")
print("   puis de restaurer le format original pour l'export/stockage.")

🔄 Workflow: Validation → Traitement → Restauration
1️⃣ Données originales:
    timestamp  revenue  users
0  2023-01-01     1000     50
1  2023-02-01     1100     55
2  2023-03-01     1050     52
3  2023-04-01     1200     60
   Index: [0, 1, 2, 3]
   Colonnes: ['timestamp', 'revenue', 'users']

2️⃣ Validation avec return_metadata=True:
            revenue  users
timestamp                 
2023-01-01     1000     50
2023-02-01     1100     55
2023-03-01     1050     52
2023-04-01     1200     60
   ✓ 'timestamp' est maintenant l'index
   Métadonnées sauvegardées: ['original_index', 'original_columns', 'index_type', 'had_time_col_in_columns', 'had_panel_cols_in_columns', 'index_was_replaced', 'time_col', 'panel_cols', 'was_sorted', 'index_name']

3️⃣ Traitement (calcul de variations):
            revenue  users  revenue_change  users_change
timestamp                                               
2023-01-01     1000     50             NaN           NaN
2023-02-01     1100     55       10

### 3.4 Vérifications pour Données Panel

Les fonctions `validate_entities_grouped()` et `validate_sorted_within_groups()` vérifient la structure des données panel.

In [9]:
# Données panel bien structurées
print("✅ Exemple 1: Données panel bien structurées")
print("="*60)
df_good = pd.DataFrame({
    'entity': ['A', 'A', 'A', 'B', 'B', 'B'],  # Groupées
    'date': pd.date_range('2023-01-01', periods=3).tolist() * 2,
    'value': [10, 20, 30, 40, 50, 60]
})
df_good = df_good.set_index(['entity', 'date'])
print(df_good)

is_grouped = validate_entities_grouped(df_good, panel_cols=['entity'])
is_sorted = validate_sorted_within_groups(df_good, panel_cols=['entity'], time_col='date')
print(f"\n✓ Entités groupées: {is_grouped}")
print(f"✓ Triées par date dans chaque groupe: {is_sorted}")

print("\n" + "="*60)

# Données panel mal structurées (entrelacées)
print("\n❌ Exemple 2: Données panel avec entités entrelacées")
print("="*60)
df_bad = pd.DataFrame({
    'entity': ['A', 'B', 'A', 'B', 'A', 'B'],  # Entrelacées!
    'date': pd.date_range('2023-01-01', periods=3).tolist() * 2,
    'value': [10, 40, 20, 50, 30, 60]
})
df_bad = df_bad.set_index(['entity', 'date'])
print(df_bad)

is_grouped_bad = validate_entities_grouped(df_bad, panel_cols=['entity'])
print(f"\n⚠️ Entités groupées: {is_grouped_bad}")
print(f"   → Les entités A et B sont entrelacées (A, B, A, B, ...)")

print("\n" + "="*60)

# Données panel non triées dans les groupes
print("\n⚠️ Exemple 3: Dates non triées dans un groupe")
print("="*60)
df_unsorted = pd.DataFrame({
    'entity': ['A', 'A', 'A'],
    'date': pd.to_datetime(['2023-03-01', '2023-01-01', '2023-02-01']),  # Non triées
    'value': [30, 10, 20]
})
df_unsorted = df_unsorted.set_index(['entity', 'date'])
print(df_unsorted)

is_sorted_unsorted = validate_sorted_within_groups(df_unsorted, panel_cols=['entity'], time_col='date')
print(f"\n⚠️ Triées par date: {is_sorted_unsorted}")
print(f"   → Les dates ne sont pas en ordre chronologique")

print("\nℹ️ Ces fonctions sont utiles pour diagnostiquer les problèmes de structure")
print("   avant d'appliquer des transformations ou des modèles.")

✅ Exemple 1: Données panel bien structurées
                   value
entity date             
A      2023-01-01     10
       2023-01-02     20
       2023-01-03     30
B      2023-01-01     40
       2023-01-02     50
       2023-01-03     60

✓ Entités groupées: True
✓ Triées par date dans chaque groupe: True


❌ Exemple 2: Données panel avec entités entrelacées
                   value
entity date             
A      2023-01-01     10
B      2023-01-02     40
A      2023-01-03     20
B      2023-01-01     50
A      2023-01-02     30
B      2023-01-03     60

⚠️ Entités groupées: False
   → Les entités A et B sont entrelacées (A, B, A, B, ...)


⚠️ Exemple 3: Dates non triées dans un groupe
                   value
entity date             
A      2023-03-01     30
       2023-01-01     10
       2023-02-01     20

⚠️ Triées par date: False
   → Les dates ne sont pas en ordre chronologique

ℹ️ Ces fonctions sont utiles pour diagnostiquer les problèmes de structure
   avant d'appliquer

## 4. Gestion des Fréquences

Le sous-module `frequency/` fournit des outils pour normaliser, convertir et aligner des données avec différentes fréquences temporelles.

### 4.1 Normalisation de Fréquences

La classe `FrequencyNormalizer` et la fonction `normalize_frequency()` standardisent les représentations de fréquences entre codes (`'M'`) et littéraux (`'monthly'`).

In [10]:
# Normalisation de fréquences
print("🔄 Normalisation de fréquences")
print("="*60)

# Différents formats d'entrée pour la même fréquence
frequencies = ['M', 'monthly', 'MONTHLY', 'month']
print("Normalisation vers code pandas:")
for freq in frequencies:
    try:
        normalized = normalize_frequency(freq)
        print(f"  '{freq:10s}' → '{normalized}'")
    except:
        print(f"  '{freq:10s}' → ❌ Non reconnu")

print("\n" + "="*60)

# Conversion vers littéraux
print("\nConversion vers littéraux:")
codes = ['D', 'W', 'M', 'Q', 'Y', 'B', 'SM']
for code in codes:
    literal = freq_to_literal(code)
    print(f"  '{code:3s}' → '{literal}'")

print("\n" + "="*60)

# Comparaison de fréquences
print("\nComparaison de fréquences (higher = plus granulaire):")
comparisons = [
    ('D', 'M'),  # Daily vs Monthly
    ('M', 'Q'),  # Monthly vs Quarterly
    ('W', 'M'),  # Weekly vs Monthly
]
for freq1, freq2 in comparisons:
    is_higher = is_higher_frequency(freq1, freq2)
    symbol = ">" if is_higher else "<"
    print(f"  {freq1} {symbol} {freq2} : {freq_to_literal(freq1)} est {'plus' if is_higher else 'moins'} granulaire")

print("\nℹ️ Fréquences disponibles: D, B, W, SM, M, Q, Y, h, min, s, ms, us, ns")

🔄 Normalisation de fréquences
Normalisation vers code pandas:
  'M         ' → 'M'
  'monthly   ' → 'M'
  'MONTHLY   ' → ❌ Non reconnu
  'month     ' → ❌ Non reconnu


Conversion vers littéraux:
  'D  ' → 'daily'
  'W  ' → 'weekly'
  'M  ' → 'monthly'
  'Q  ' → 'quarterly'
  'Y  ' → 'annual'
  'B  ' → 'business_daily'
  'SM ' → 'semi_monthly'


Comparaison de fréquences (higher = plus granulaire):
  D > M : daily est plus granulaire
  M > Q : monthly est plus granulaire
  W > M : weekly est plus granulaire

ℹ️ Fréquences disponibles: D, B, W, SM, M, Q, Y, h, min, s, ms, us, ns


### 4.2 Conversion Descendante (Agrégation)

Quand on passe d'une haute fréquence vers une basse fréquence (ex: daily → monthly), on agrège les données. **L'impact de la méthode d'agrégation est significatif** selon le type de données.

In [11]:
# Données journalières à agréger
print("📊 Agrégation: Daily → Monthly")
print("="*60)

# Création de données journalières (janvier 2023)
daily_dates = pd.date_range('2023-01-01', '2023-01-31', freq='D')
np.random.seed(42)
daily_sales = pd.Series(
    np.random.randint(50, 150, len(daily_dates)),
    index=daily_dates,
    name='daily_sales'
)
print(f"Données journalières: {len(daily_sales)} jours")
print(daily_sales.head(7))
print(f"Total janvier: {daily_sales.sum()}")

print("\n" + "="*60)

# Impact des différentes méthodes d'agrégation
print("\n⚙️ Impact de la méthode d'agrégation (DIFFÉRENCES IMPORTANTES):")
print("-"*60)

converter = FrequencyConverter()

# Méthode 1: mean (moyenne)
monthly_mean = converter.aggregate_to_lower_frequency(
    daily_sales, 
    target_freq='M', 
    method='mean'
)
print(f"📍 mean  : {monthly_mean.values[0]:.2f} (moyenne des ventes journalières)")

# Méthode 2: sum (somme)
monthly_sum = converter.aggregate_to_lower_frequency(
    daily_sales, 
    target_freq='M', 
    method='sum'
)
print(f"📍 sum   : {monthly_sum.values[0]:.2f} (total des ventes du mois)")

# Méthode 3: first (première valeur)
monthly_first = converter.aggregate_to_lower_frequency(
    daily_sales, 
    target_freq='M', 
    method='first'
)
print(f"📍 first : {monthly_first.values[0]:.2f} (ventes du 1er janvier)")

print("\n💡 Choix de la méthode:")
print("   - mean  : Variables intensives (taux, températures, prix moyens)")
print("   - sum   : Variables extensives (ventes totales, production totale)")
print("   - first : Valeurs de début de période (stocks en début de mois)")
print("   - last  : Valeurs de fin de période (stocks en fin de mois)")

print("\nℹ️ Autres méthodes disponibles: median, min, max, std, count")

print("\n" + "="*60)

# Exemple avec DataFrame
print("\n📊 Agrégation d'un DataFrame multi-colonnes:")
print("-"*60)
df_daily = pd.DataFrame({
    'revenue': daily_sales.values,
    'costs': daily_sales.values * 0.7,
    'margin_rate': (daily_sales.values - daily_sales.values * 0.7) / daily_sales.values * 100
}, index=daily_dates)
print("Données journalières (échantillon):")
print(df_daily.head(3))

# Agrégation avec méthodes différentes par colonne
df_monthly = converter.aggregate_to_lower_frequency(
    df_daily[['revenue', 'costs']], 
    target_freq='M',
    method='sum'  # Total pour revenus et coûts
)
# Pour le taux de marge, on recalcule
df_monthly['margin_rate'] = (df_monthly['revenue'] - df_monthly['costs']) / df_monthly['revenue'] * 100
print("\nAgrégé mensuellement:")
print(df_monthly)

📊 Agrégation: Daily → Monthly
Données journalières: 31 jours
2023-01-01    101
2023-01-02    142
2023-01-03     64
2023-01-04    121
2023-01-05    110
2023-01-06     70
2023-01-07    132
Freq: D, Name: daily_sales, dtype: int32
Total janvier: 3166


⚙️ Impact de la méthode d'agrégation (DIFFÉRENCES IMPORTANTES):
------------------------------------------------------------
📍 mean  : 102.13 (moyenne des ventes journalières)
📍 sum   : 3166.00 (total des ventes du mois)
📍 first : 101.00 (ventes du 1er janvier)

💡 Choix de la méthode:
   - mean  : Variables intensives (taux, températures, prix moyens)
   - sum   : Variables extensives (ventes totales, production totale)
   - first : Valeurs de début de période (stocks en début de mois)
   - last  : Valeurs de fin de période (stocks en fin de mois)

ℹ️ Autres méthodes disponibles: median, min, max, std, count


📊 Agrégation d'un DataFrame multi-colonnes:
------------------------------------------------------------
Données journalières (échan

### 4.3 Conversion Ascendante (Interpolation)

Quand on passe d'une basse fréquence vers une haute fréquence (ex: monthly → daily), on interpole les valeurs intermédiaires.

In [12]:
# Données mensuelles à interpoler
print("📈 Interpolation: Monthly → Daily")
print("="*60)

# Création de données mensuelles (5 mois)
monthly_dates = pd.date_range('2022-12-01', '2023-04-01', freq='MS')
monthly_values = pd.Series([100, 100, 150, 120, 120], index=monthly_dates, name='indicator')
print("Données mensuelles:")
print(monthly_values)

print("\n" + "="*60)

# Interpolation linéaire
print("\n⚙️ Méthode 'linear' (interpolation linéaire):")
print("-"*60)
daily_linear = converter.interpolate_to_higher_frequency(
    monthly_values,
    target_freq='D',
    method='linear'
)
print(f"Interpolé sur {len(daily_linear)} jours")
print("Échantillon (premiers et derniers jours de janvier):")
print(daily_linear.iloc[:3])
print("...")
print(daily_linear.iloc[28:32])
print("→ Transition lisse et linéaire entre les points")

print("\n" + "="*60)

# Interpolation cubique
print("\n⚙️ Méthode 'cubic' (interpolation cubique - plus lisse):")
print("-"*60)
daily_cubic = converter.interpolate_to_higher_frequency(
    monthly_values,
    target_freq='D',
    method='cubic'
)
print("Échantillon (milieu de période):")
print(daily_cubic.iloc[13:17])
print("→ Courbe plus lisse, peut dépasser les bornes")

print("\n💡 Choix de la méthode:")
print("   - linear : Simple, garantit de rester entre les bornes")
print("   - cubic  : Plus lisse, bon pour des tendances continues")
print("   - time   : Tient compte du temps réel entre les observations")

print("\nℹ️ Autres méthodes: nearest, zero, slinear, quadratic")
print("   et fill_method pour gérer les valeurs manquantes: ffill, bfill")

print("\n" + "="*60)

# Comparaison visuelle des valeurs
print("\n📊 Comparaison des méthodes (jour 15):")
print("-"*60)
day_15_idx = 14  # Index du 15ème jour
print(f"Linéaire : {daily_linear.iloc[day_15_idx]:.2f}")
print(f"Cubique  : {daily_cubic.iloc[day_15_idx]:.2f}")
print(f"Différence: {abs(daily_linear.iloc[day_15_idx] - daily_cubic.iloc[day_15_idx]):.2f}")
print("\n→ La différence peut être significative selon la courbure des données")

📈 Interpolation: Monthly → Daily
Données mensuelles:
2022-12-01    100
2023-01-01    100
2023-02-01    150
2023-03-01    120
2023-04-01    120
Freq: MS, Name: indicator, dtype: int64


⚙️ Méthode 'linear' (interpolation linéaire):
------------------------------------------------------------
Interpolé sur 151 jours
Échantillon (premiers et derniers jours de janvier):
2022-12-01    100.0
2022-12-02    100.0
2022-12-03    100.0
Freq: D, Name: indicator, dtype: float64
...
2022-12-29    100.0
2022-12-30    100.0
2022-12-31    100.0
2023-01-01    100.0
Freq: D, Name: indicator, dtype: float64
→ Transition lisse et linéaire entre les points


⚙️ Méthode 'cubic' (interpolation cubique - plus lisse):
------------------------------------------------------------
Échantillon (milieu de période):
2022-12-14    81.014940
2022-12-15    80.958126
2022-12-16    81.068076
2022-12-17    81.338043
Freq: D, Name: indicator, dtype: float64
→ Courbe plus lisse, peut dépasser les bornes

💡 Choix de la méthod

### 4.4 Alignement de Datasets Multi-Fréquences

La méthode `align_frequencies()` permet d'harmoniser plusieurs datasets avec des fréquences différentes vers une fréquence cible commune.

In [13]:
# Datasets avec différentes fréquences
print("🔗 Alignement de datasets multi-fréquences")
print("="*60)

# Dataset 1: Données trimestrielles (PIB)
quarterly_dates = pd.date_range('2023-01-01', periods=8, freq='QS')
gdp_quarterly = pd.Series([2000, 2050, 2100, 2150, 2000, 2050, 2100, 2150], index=quarterly_dates, name='gdp')
print("📊 Dataset 1: PIB Trimestriel")
print(gdp_quarterly)

# Dataset 2: Données mensuelles (Inflation)
monthly_dates = pd.date_range('2023-01-01', periods=24, freq='MS')
inflation_monthly = pd.Series(
    [2.0, 2.1, 2.2, 2.3, 2.2, 2.4, 2.5, 2.6, 2.5, 2.7, 2.8, 2.9, 2.0, 2.1, 2.2, 2.3, 2.2, 2.4, 2.5, 2.6, 2.5, 2.7, 2.8, 2.9],
    index=monthly_dates,
    name='inflation'
)
print("\n📊 Dataset 2: Inflation Mensuelle")
print(inflation_monthly.head(6))

# Dataset 3: Données annuelles (Population)
annual_dates = pd.date_range('2023-01-01', periods=2, freq='YS')
population_annual = pd.Series([67.0, 67.0], index=annual_dates, name='population_millions')
print("\n📊 Dataset 3: Population Annuelle")
print(population_annual)

print("\n" + "="*60)

# Alignement vers fréquence mensuelle
print("\n🎯 Alignement vers fréquence mensuelle:")
print("-"*60)
aligned = converter.align_frequencies(
    gdp_quarterly,
    inflation_monthly,
    population_annual,
    target_freq='M',
    method='linear'  # Interpolation pour upsampling
)
print(f"Résultat: {len(aligned[0])} mois")
print("\nDataFrame aligné (échantillon):")
print(aligned[0].head(6))
print("\n✓ Toutes les variables sont maintenant à la même fréquence mensuelle")
print("  - PIB: interpolé linéairement entre les trimestres")
print("  - Inflation: fréquence native préservée")
print("  - Population: répliquée sur tous les mois")

🔗 Alignement de datasets multi-fréquences
📊 Dataset 1: PIB Trimestriel
2023-01-01    2000
2023-04-01    2050
2023-07-01    2100
2023-10-01    2150
2024-01-01    2000
2024-04-01    2050
2024-07-01    2100
2024-10-01    2150
Freq: QS-JAN, Name: gdp, dtype: int64

📊 Dataset 2: Inflation Mensuelle
2023-01-01    2.0
2023-02-01    2.1
2023-03-01    2.2
2023-04-01    2.3
2023-05-01    2.2
2023-06-01    2.4
Freq: MS, Name: inflation, dtype: float64

📊 Dataset 3: Population Annuelle
2023-01-01    67.0
2024-01-01    67.0
Freq: YS-JAN, Name: population_millions, dtype: float64


🎯 Alignement vers fréquence mensuelle:
------------------------------------------------------------
Résultat: 24 mois

DataFrame aligné (échantillon):
2023-01-01    2000.000000
2023-02-01    2016.666667
2023-03-01    2033.333333
2023-04-01    2050.000000
2023-05-01    2066.666667
2023-06-01    2083.333333
Freq: MS, Name: gdp, dtype: float64

✓ Toutes les variables sont maintenant à la même fréquence mensuelle
  - PIB: int

## 5. Gestion des Durées

Le sous-module `duration/` permet de normaliser et convertir des durées entre différentes unités temporelles.

### 5.1 Normalisation et Conversion de Durées

Les classes `DurationNormalizer` et `DurationConverter` permettent de standardiser et convertir les durées.

In [14]:
# Normalisation de durées
print("🕐 Normalisation et conversion de durées")
print("="*60)

# Normalisation
print("📋 Normalisation de durées:")
durations = ['h', 'hour', 'hourly', 'D', 'day']
for dur in durations:
    try:
        normalized = normalize_duration(dur)
        literal = duration_to_literal(normalized)
        print(f"  '{dur:10s}' → code: '{normalized:3s}' → littéral: '{literal}'")
    except:
        print(f"  '{dur:10s}' → ❌ Non reconnu")

print("\n" + "="*60)

# Conversion de durées
print("\n🔄 Conversion entre unités:")
print("-"*60)

# Exemple: Convertir 5 jours en différentes unités
days = 5
print(f"Valeur de base: {days} jours\n")

dur_converter = DurationConverter()

conversions = [
    ('D', 'h', 'heures'),
    ('D', 'min', 'minutes'),
    ('D', 'M', 'mois (approx)'),
]

for from_unit, to_unit, label in conversions:
    converted = dur_converter.convert(days, from_unit, to_unit)
    print(f"  {days} jours = {converted:.2f} {label}")

print("\n" + "="*60)

# Impact du rounding (IMPORTANT)
print("\n⚙️ Impact du paramètre 'rounding' (DIFFÉRENCE IMPORTANTE):")
print("-"*60)

# Conversion avec arrondis: 37 heures en jours
hours = 37
print(f"Valeur de base: {hours} heures\n")

# Sans arrondi (valeur exacte)
exact = dur_converter.convert(hours, 'h', 'D', rounding=None)
print(f"  rounding=None  : {exact:.4f} jours (valeur exacte)")

# Avec floor (arrondi inférieur)
floor = dur_converter.convert(hours, 'h', 'D', rounding='floor')
print(f"  rounding=floor : {floor:.0f} jours (arrondi inférieur)")

# Avec ceil (arrondi supérieur)
ceil_val = dur_converter.convert(hours, 'h', 'D', rounding='ceil')
print(f"  rounding=ceil  : {ceil_val:.0f} jours (arrondi supérieur)")

print("\n💡 Utilisation du rounding:")
print("   - None  : Calculs précis, analyses scientifiques")
print("   - floor : Garanties minimales (ex: 'au moins X jours')")
print("   - ceil  : Garanties maximales (ex: 'peut prendre jusqu'à X jours')")

print("\n" + "="*60)

# Exemple pratique: calcul de délais
print("\n📊 Exemple pratique: Calcul de délais de livraison")
print("-"*60)

# Délai de 72 heures → conversion en jours
delivery_hours = 72
delivery_days_exact = dur_converter.convert(delivery_hours, 'h', 'D')
delivery_days_ceil = dur_converter.convert(delivery_hours, 'h', 'D', rounding='ceil')

print(f"Délai annoncé: {delivery_hours} heures")
print(f"  Exact: {delivery_days_exact:.1f} jours")
print(f"  Avec marge (ceil): {delivery_days_ceil:.0f} jours")
print(f"\n→ Pour la communication client, on dirait: 'Livraison sous {delivery_days_ceil:.0f} jours'")

print("\nℹ️ Unités disponibles: ns, us, ms, s, min, h, D, B, W, M, Q, Y")

🕐 Normalisation et conversion de durées
📋 Normalisation de durées:
  'h         ' → code: 'h  ' → littéral: 'hour'
  'hour      ' → code: 'h  ' → littéral: 'hour'
  'hourly    ' → ❌ Non reconnu
  'D         ' → code: 'D  ' → littéral: 'day'
  'day       ' → code: 'D  ' → littéral: 'day'


🔄 Conversion entre unités:
------------------------------------------------------------
Valeur de base: 5 jours

  5 jours = 120.00 heures
  5 jours = 7200.00 minutes
  5 jours = 0.17 mois (approx)


⚙️ Impact du paramètre 'rounding' (DIFFÉRENCE IMPORTANTE):
------------------------------------------------------------
Valeur de base: 37 heures

  rounding=None  : 1.5417 jours (valeur exacte)
  rounding=floor : 1 jours (arrondi inférieur)
  rounding=ceil  : 2 jours (arrondi supérieur)

💡 Utilisation du rounding:
   - None  : Calculs précis, analyses scientifiques
   - floor : Garanties minimales (ex: 'au moins X jours')
   - ceil  : Garanties maximales (ex: 'peut prendre jusqu'à X jours')


📊 Exemple p

## 6. Positions de Périodes

Le sous-module `position/` gère les positions au sein des périodes (début vs fin) et la manipulation des offsets pandas (MS, ME, QS, QE, etc.).

### 6.1 Normalisation et Conversion de Positions

Les classes `PeriodPositionNormalizer` et `PeriodPositionConverter` gèrent les positions start (S) et end (E) des périodes.

In [15]:
# Normalisation de positions
print("📍 Positions de périodes: Start vs End")
print("="*60)

# Normalizer et Converter
pos_normalizer = PeriodPositionNormalizer()
pos_converter = PeriodPositionConverter()

# Normalisation
print("📋 Normalisation de positions:")
positions = ['S', 'start', 'E', 'end']
for pos in positions:
    normalized = pos_normalizer.normalize(pos)
    literal = pos_normalizer.to_literal(normalized)
    print(f"  '{pos:6s}' → code: '{normalized}' → littéral: '{literal}'")

print("\n" + "="*60)

# Décomposition d'offsets pandas
print("\n🔧 Décomposition d'offsets pandas:")
print("-"*60)
offsets = ['MS', 'ME', 'QS', 'QE', 'YS', 'YE']
for offset in offsets:
    freq, pos = pos_normalizer.decompose_offset(offset)
    pos_literal = pos_normalizer.to_literal(pos)
    print(f"  '{offset}' → fréquence: '{freq}', position: '{pos}' ({pos_literal})")

print("\n" + "="*60)

# Conversion d'offsets (MS → ME)
print("\n🔄 Conversion d'offsets (changement de position):")
print("-"*60)
print("Conversion de 'start' vers 'end':")
conversions = [('MS', 'E'), ('QS', 'E'), ('YS', 'E')]
for offset, target_pos in conversions:
    converted = pos_converter.convert_offset(offset, target_pos)
    print(f"  {offset} → {converted}")

print("\n" + "="*60)

# Impact sur les séries temporelles
print("\n⚙️ Impact sur l'alignement des séries temporelles:")
print("-"*60)

# Série avec dates de début de mois (MS)
dates_start = pd.date_range('2023-01-01', periods=4, freq='MS')
values = [100, 110, 105, 115]
ts_start = pd.Series(values, index=dates_start, name='sales')
print("Série avec dates de début de mois (MS):")
print(ts_start)

# Conversion vers dates de fin de mois (ME)
dates_end = pd.date_range('2023-01-31', periods=4, freq='ME')
ts_end = pd.Series(values, index=dates_end, name='sales')
print("\nSérie avec dates de fin de mois (ME):")
print(ts_end)

print("\n💡 Utilisation:")
print("   - Start (MS, QS, YS): Pour données mesurées en début de période")
print("   - End (ME, QE, YE): Pour données mesurées en fin de période")
print("   - Impact sur le merge/join de datasets avec différentes conventions")

print("\n" + "="*60)

# Combinaison fréquence + position
print("\n🎯 Combinaison fréquence + position:")
print("-"*60)
combinations = [
    ('M', 'S'),  # Monthly Start
    ('M', 'E'),  # Monthly End
    ('Q', 'S'),  # Quarterly Start
    ('Q', 'E'),  # Quarterly End
]
for freq, pos in combinations:
    offset = pos_normalizer.combine_frequency_position(freq, pos)
    print(f"  Fréquence '{freq}' + Position '{pos}' → Offset '{offset}'")

📍 Positions de périodes: Start vs End
📋 Normalisation de positions:
  'S     ' → code: 'S' → littéral: 'start'
  'start ' → code: 'S' → littéral: 'start'
  'E     ' → code: 'E' → littéral: 'end'
  'end   ' → code: 'E' → littéral: 'end'


🔧 Décomposition d'offsets pandas:
------------------------------------------------------------
  'MS' → fréquence: 'M', position: 'S' (start)
  'ME' → fréquence: 'M', position: 'E' (end)
  'QS' → fréquence: 'Q', position: 'S' (start)
  'QE' → fréquence: 'Q', position: 'E' (end)
  'YS' → fréquence: 'Y', position: 'S' (start)
  'YE' → fréquence: 'Y', position: 'E' (end)


🔄 Conversion d'offsets (changement de position):
------------------------------------------------------------
Conversion de 'start' vers 'end':
  MS → ME
  QS → QE
  YS → YE


⚙️ Impact sur l'alignement des séries temporelles:
------------------------------------------------------------
Série avec dates de début de mois (MS):
2023-01-01    100
2023-02-01    110
2023-03-01    105
2023-04

## 8. Exemple Complet End-to-End

Cet exemple combine plusieurs composants du module `utils` pour illustrer un workflow réaliste de préparation de données multi-fréquences.

### 8.1 Contexte et Objectif

**Scénario:** Nous avons trois sources de données économiques avec différentes fréquences:
1. Ventes journalières (3 magasins)
2. Indicateur économique mensuel  
3. Taux directeur trimestriel de la banque centrale

**Objectif:** Préparer un dataset mensuel consolidé pour une analyse économétrique.

In [18]:
print("="*70)
print(" WORKFLOW COMPLET: PRÉPARATION DE DONNÉES MULTI-FRÉQUENCES")
print("="*70)

# ============================================================================
# ÉTAPE 1: CHARGEMENT DES DONNÉES BRUTES (format désordonnié)
# ============================================================================
print("\n📁 ÉTAPE 1: Chargement des données brutes")
print("-"*70)

# Source 1: Ventes journalières (panel data, format CSV)
# Données étendues sur 3 mois pour assurer plusieurs observations après agrégation mensuelle
dates_jan = ['2023-01-10', '2023-01-15', '2023-01-20']
dates_feb = ['2023-02-05', '2023-02-12', '2023-02-25']
dates_mar = ['2023-03-08', '2023-03-18', '2023-03-28']
all_dates = dates_jan + dates_feb + dates_mar

df_sales_raw = pd.DataFrame({
    'date_str': all_dates * 3,  # Non triées!
    'store': ['A'] * 9 + ['B'] * 9 + ['C'] * 9,
    'daily_sales': [100, 110, 95, 200, 210, 195, 150, 160, 145] * 3
})
print("Ventes journalières (format brut):")
print(df_sales_raw.head(6))
print(f"  ⚠️ Dates non triées, format string")

# Source 2: Indicateur économique mensuel
df_econ_raw = pd.DataFrame({
    'month': ['2023-01', '2023-02', '2023-03'],
    'economic_index': [102.5, 103.1, 102.8]
})
print("\nIndicateur économique mensuel:")
print(df_econ_raw)

# Source 3: Taux directeur trimestriel (données étendues pour interpolation)
# Note: Au minimum 2 observations requises pour interpolation linéaire
df_rate_raw = pd.DataFrame({
    'quarter': ['2023Q1', '2023Q2', '2023Q3', '2023Q4'],
    'interest_rate': [4.5, 4.75, 5.0, 5.25]
})
print("\nTaux directeur trimestriel:")
print(df_rate_raw)

# ============================================================================
# ÉTAPE 2: VALIDATION ET NETTOYAGE
# ============================================================================
print("\n" + "="*70)
print("🔍 ÉTAPE 2: Validation et nettoyage des données")
print("-"*70)

# Validation des ventes (panel data)
df_sales_validated, metadata_sales = validate_temporal_data(
    df_sales_raw,
    time_col='date_str',
    panel_cols=['store'],
    sort_data=True,  # Tri automatique
    return_metadata=True,
    strict=False
)
print("✓ Ventes validées (panel, triées):")
print(df_sales_validated.head(6))

# Validation de l'indicateur économique
df_econ_validated = validate_temporal_data(
    df_econ_raw,
    time_col='month'
)
print("\n✓ Indicateur économique validé:")
print(df_econ_validated)

# Validation du taux directeur (format trimestriel spécial)
# Conversion des étiquettes trimestrielles vers datetime
df_rate_raw_expanded = df_rate_raw.copy()
# Conversion: '2023Q1' -> début du trimestre ('2023-01-01'), '2023Q2' -> '2023-04-01', etc.
df_rate_raw_expanded['quarter_date'] = df_rate_raw_expanded['quarter'].apply(
    lambda q: pd.to_datetime(f"{q[:4]}-{(int(q[-1])-1)*3 + 1:02d}-01")
)
df_rate_validated = pd.Series(
    df_rate_raw_expanded['interest_rate'].values,
    index=df_rate_raw_expanded['quarter_date'],
    name='interest_rate'
)
print("\n✓ Taux directeur validé:")
print(df_rate_validated)

# ============================================================================
# ÉTAPE 3: CONVERSION DE FRÉQUENCES
# ============================================================================
print("\n" + "="*70)
print("📈 ÉTAPE 3: Conversion vers fréquence mensuelle cible")
print("-"*70)

freq_converter = FrequencyConverter()

# 3a. Agrégation des ventes journalières → mensuelles (par magasin)
print("Agrégation: Daily → Monthly (par magasin)")
monthly_sales_by_store = {}
for store in df_sales_validated.index.get_level_values('store').unique():
    store_data = df_sales_validated.loc[store]
    monthly = freq_converter.aggregate_to_lower_frequency(
        store_data['daily_sales'],
        target_freq='M',
        method='sum'  # Total des ventes par mois
    )
    monthly_sales_by_store[f'sales_{store}'] = monthly

df_sales_monthly = pd.DataFrame(monthly_sales_by_store)
print("✓ Ventes mensuelles par magasin:")
print(df_sales_monthly)

# 3b. Indicateur économique: déjà mensuel, on garde tel quel
print("\n✓ Indicateur économique: déjà mensuel")

# 3c. Interpolation du taux directeur trimestriel → mensuel
print("\nInterpolation: Quarterly → Monthly")
df_rate_monthly = freq_converter.interpolate_to_higher_frequency(
    df_rate_validated,
    target_freq='MS',
    method='linear'
)
print("✓ Taux directeur mensuel (interpolé):")
print(df_rate_monthly)

# ============================================================================
# ÉTAPE 4: ALIGNEMENT ET CONSOLIDATION
# ============================================================================
print("\n" + "="*70)
print("🎯 ÉTAPE 4: Alignement et consolidation")
print("-"*70)

# Alignement de tous les datasets vers fréquence mensuelle commune
aligned_data = freq_converter.align_frequencies(
    df_sales_monthly,
    df_econ_validated,
    df_rate_monthly,
    target_freq='M',
)
print("✓ Données alignées à fréquence mensuelle:")
print(aligned_data)

# ============================================================================
# ÉTAPE 5: ENRICHISSEMENT ET CALCULS
# ============================================================================
print("\n" + "="*70)
print("📊 ÉTAPE 5: Enrichissement avec calculs dérivés")
print("-"*70)

# Calcul du total des ventes (tous magasins)
sales_data = aligned_data[0]
sales_data['total_sales'] = sales_data[[c for c in sales_data.columns if c.startswith('sales_')]].sum(axis=1)

# Calcul de la variation mensuelle de l'indicateur économique
sales_data['econ_change'] = aligned_data[1]['economic_index'].pct_change() * 100

# Extraction de features temporelles
sales_data['month'] = sales_data.index.month
sales_data['quarter'] = sales_data.index.quarter

print("✓ Dataset final enrichi:")
print(sales_data)

# ============================================================================
# ÉTAPE 6: EXPORT (avec restauration de structure si nécessaire)
# ============================================================================
print("\n" + "="*70)
print("💾 ÉTAPE 6: Préparation pour export")
print("-"*70)

# Conversion de l'index datetime vers string pour export CSV
final_for_export = sales_data.copy()
final_for_export.index = final_for_export.index.strftime('%Y-%m')
print("✓ Dataset prêt pour export (index au format string):")
print(final_for_export)

print("\n" + "="*70)
print("✅ WORKFLOW TERMINÉ!")
print("="*70)
print("\nRésumé des opérations:")
print("  1. ✓ Validation de 3 sources de données (différents formats)")
print("  2. ✓ Conversion de fréquences (Daily → Monthly, Quarterly → Monthly)")
print("  3. ✓ Alignement et consolidation")
print("  4. ✓ Enrichissement avec calculs dérivés")
print("  5. ✓ Préparation pour export")
print("\n→ Dataset final prêt pour analyse économétrique!")

 WORKFLOW COMPLET: PRÉPARATION DE DONNÉES MULTI-FRÉQUENCES

📁 ÉTAPE 1: Chargement des données brutes
----------------------------------------------------------------------
Ventes journalières (format brut):
     date_str store  daily_sales
0  2023-01-10     A          100
1  2023-01-15     A          110
2  2023-01-20     A           95
3  2023-02-05     A          200
4  2023-02-12     A          210
5  2023-02-25     A          195
  ⚠️ Dates non triées, format string

Indicateur économique mensuel:
     month  economic_index
0  2023-01           102.5
1  2023-02           103.1
2  2023-03           102.8

Taux directeur trimestriel:
  quarter  interest_rate
0  2023Q1           4.50
1  2023Q2           4.75
2  2023Q3           5.00
3  2023Q4           5.25

🔍 ÉTAPE 2: Validation et nettoyage des données
----------------------------------------------------------------------
✓ Ventes validées (panel, triées):
                  daily_sales
store date_str               
A     2023-01-10 

## Conclusion

Ce notebook a couvert de manière exhaustive les fonctionnalités du module `tsforecast.utils`:

### Modules Principaux

1. **time.py** - Manipulation des dates et périodes
2. **validation.py** - Validation et préparation des données temporelles et panel
3. **frequency/** - Normalisation, conversion et alignement de fréquences
4. **duration/** - Normalisation et conversion de durées
5. **position/** - Gestion des positions dans les périodes (start/end)
6. **base_transformers.py** - Classes de base pour transformers sklearn-compatible

### Bonnes Pratiques

- **Toujours valider** les données avec `validate_temporal_data()` avant traitement
- **Choisir la méthode appropriée** pour agrégation (mean vs sum) selon le type de variable
- **Utiliser l'interpolation avec précaution** - linear est sûr, cubic peut dépasser les bornes
- **Sauvegarder les métadonnées** avec `return_metadata=True` pour restauration ultérieure
- **Normaliser les fréquences** pour éviter les ambiguïtés de représentation

### Ressources

Pour plus d'informations sur les autres modules du package `tsforecast`:
- Cross-validation temporelle: voir notebook correspondant
- Delays (retards de publication): voir notebook correspondant
- Modèles et prédictions: voir documentation du package